# Predict TIMSS item parameters

Predict the IRT parameters of new TIMSS mathematics items from the most similar released items, whose parameters are known.

1. Index the released items for keyword, semantic, and hybrid retrieval, and add each item's calibrated parameters to its stored metadata.
2. Predict a new item's parameters with an LLM that gets the most similar calibrated items of the same grade as context.
3. Check the method: predict every calibrated released item as if it were new, compare each prediction with the item's calibrated parameters, and summarize the prediction errors by grade.

In [1]:
import concurrent.futures
import functools
import hashlib
import json
import os
import re
import time
import zlib
from collections import Counter, deque
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal
from uuid import NAMESPACE_URL, uuid5

import numpy as np
import openai
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from pydantic import BaseModel, Field, ValidationError, create_model, model_validator
from qdrant_client import QdrantClient, models

load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

DATA_DIR = Path.cwd() / "data"
RELEASED_ITEMS_PATH = DATA_DIR / "timss_math_released_items.jsonl"
ITEM_PARAMETERS_PATH = DATA_DIR / "timss_math_released_item_parameters.xlsx"

## Index the released items for retrieval

### One chunk per item

Every item in `data/timss_math_released_items.jsonl` (the released TIMSS and TIMSS Advanced mathematics items of 1995 to 2011, built by `get_data.ipynb`) is one chunk, except the three duplicate texts below.
The chunk's `text` is the stem followed by one line per option, which is what students read.
It is the only text that is embedded and keyword-indexed.
The correct option, scoring guide, and scoring notes are never part of it, and neither are the model-written figure descriptions or the accessibility text.

Each chunk also carries the metadata in `METADATA_FIELDS` for filters and display: item ID and label, assessment, grade, subject, content, topic, and cognitive domains, item type, maximum points, whether the item has a figure, and percent correct.
Answers and scoring guides stay out of the index altogether; look them up in the released-items JSONL by `item_key` (`items_by_key`).
Item IDs alone repeat across assessments (K1 is in both TIMSS 1995 and TIMSS Advanced 1995), so `item_key` adds the assessment, grade, and subject.

### Duplicate texts left out

The 731 items have 728 distinct texts, because three items repeat another item's stem and options word for word.
The three repeats are left out of the index (`DUPLICATE_ITEMS`), so the collection holds 728 items and every text once:

| Left out | Same text as | Why the text repeats |
| --- | --- | --- |
| K7, TIMSS 1995 grade 4 | M012030, TIMSS 2003 grade 8 | TIMSS 1995 gave the item to both populations, and TIMSS 2003 gave it again at grade 8. |
| D11, TIMSS 1999 grade 8 | M012023, TIMSS 2003 grade 4 | M012023 is D11's permanent ID; TIMSS 2003 gave the item at grade 4. |
| L15B, TIMSS Advanced 1995 | L15A, TIMSS Advanced 1995 | The booklet prints the whole item on the pages of both parts, so both parts have the same text. |

A repeat has the same vectors as the item it repeats, so the two always tie and take two result places for one text.
For parameter prediction, leaving the repeats out means:

- K7 (calibrated at grade 4) and D11 (calibrated at grade 8) are neither reference items nor predicted. The items they repeat stay, with the parameters of their own grades, and similar items are only looked for within an item's grade, so the grade 4 parameters of K7 and the grade 8 parameters of D11 are not used.
- L15A and L15B have no parameters (the 1995-2008 rescaling left them out), so leaving out L15B changes no prediction.

The cell checks that each repeat still has the same text as the item it repeats, and that no other text repeats.
Three cross-grade pairs have nearly, but not exactly, the same text and are kept: H08 (TIMSS 1999 grade 8) and M012044 (TIMSS 2003 grade 4), and H12 and M012048, are the same items under booklet and permanent IDs with slightly different transcriptions, and M031334 (TIMSS 2007 grade 4) and M032273 (TIMSS 2007 grade 8) are one item with options B and C swapped.
Each copy is calibrated at its own grade, and same-grade retrieval never puts the two copies in each other's context.

In [2]:
# Copied into each chunk for filters and display. Answers and scoring (correct_option, scoring_guide, scoring_notes)
# stay out of the index altogether.
METADATA_FIELDS = [
    "item_id",
    "item_label",
    "assessment",
    "grade",
    "subject",
    "content_domain",
    "main_topic",
    "cognitive_domain",
    "item_type",
    "max_points",
    "has_figure",
    "pct_correct_intl_avg",
    "pct_correct_usa",
]
# Items whose stem and options repeat another item's word for word, each with the item it repeats. They are left out
# of the index.
DUPLICATE_ITEMS = {
    "TIMSS 1995/grade 4/Mathematics/K7": "TIMSS 2003/grade 8/Mathematics/M012030",
    "TIMSS 1999/grade 8/Mathematics/D11": "TIMSS 2003/grade 4/Mathematics/M012023",
    "TIMSS Advanced 1995/grade 12/Advanced Mathematics/L15B": "TIMSS Advanced 1995/grade 12/Advanced Mathematics/L15A",
}


def item_key(item):
    """Unique key of an item, such as "TIMSS 1995/grade 4/Mathematics/K1"; item IDs alone repeat across assessments."""
    return f"{item['assessment']}/grade {item['grade']}/{item['subject']}/{item['item_id']}"


def item_text(item):
    """Text to embed and keyword-index: the stem, then one line per option. Never the answer or scoring."""
    options = "\n".join(f"{option['label']}. {option['text']}" for option in item["options"] or [])
    return "\n\n".join(part for part in (item["stem"], options) if part)


released_items = [json.loads(line) for line in RELEASED_ITEMS_PATH.read_text(encoding="utf-8").splitlines()]
items_by_key = {item_key(item): item for item in released_items}
if len(items_by_key) != len(released_items):
    raise ValueError("Item keys must be unique")
for duplicate, original in DUPLICATE_ITEMS.items():
    if item_text(items_by_key[duplicate]) != item_text(items_by_key[original]):
        raise ValueError(f"{duplicate} no longer repeats the text of {original}; update DUPLICATE_ITEMS")
chunks = [
    {"item_key": key, **{field: item[field] for field in METADATA_FIELDS}, "text": item_text(item)}
    for key, item in items_by_key.items()
    if key not in DUPLICATE_ITEMS
]

chunks_table = pd.DataFrame(chunks)
if not chunks_table["text"].str.strip().astype(bool).all():
    raise ValueError("Every chunk needs text")
repeated_texts = chunks_table[chunks_table["text"].duplicated(keep=False)]
if not repeated_texts.empty:
    raise ValueError(f"Texts repeat in {repeated_texts['item_key'].tolist()}; add the repeats to DUPLICATE_ITEMS")
print(f"{len(chunks)} chunks: {len(released_items)} items, {len(DUPLICATE_ITEMS)} duplicate texts left out")
display(chunks_table.groupby(["assessment", "grade", "subject"]).size().rename("items").to_frame())
display(chunks_table["text"].str.len().describe().round().rename("text characters").to_frame().T)
display(chunks_table.head())

728 chunks: 731 items, 3 duplicate texts left out


items
assessment          grade subject                    
TIMSS 1995          4     Mathematics              69
TIMSS 1999          8     Mathematics              81
TIMSS 2003          4     Mathematics              79
                    8     Mathematics              99
TIMSS 2007          4     Mathematics              74
                    8     Mathematics              89
TIMSS 2011          4     Mathematics              73
                    8     Mathematics              88
TIMSS Advanced 1995 12    Advanced Mathematics     36
TIMSS Advanced 2008 12    Advanced Mathematics     40

,count,mean,std,min,25%,50%,75%,max
text characters,728.0,250.0,230.0,22.0,124.0,195.0,294.0,2451.0


,item_key,item_id,item_label,assessment,grade,subject,content_domain,main_topic,cognitive_domain,item_type,max_points,has_figure,pct_correct_intl_avg,pct_correct_usa,text
0,TIMSS 1995/grade 4/Mathematics/I3,I3,Which number is it,TIMSS 1995,4,Mathematics,Whole Numbers,NaN,Using Complex Procedures,multiple_choice,1,False,57.0,57.0,When you subtract one of the numbers below fro...
1,TIMSS 1995/grade 4/Mathematics/I4,I4,What is 3 times 23,TIMSS 1995,4,Mathematics,Whole Numbers,NaN,Performing Routine Procedures,multiple_choice,1,False,84.0,90.0,What is 3 times 23?\n\nA. 323\nB. 233\nC. 69\n...
2,TIMSS 1995/grade 4/Mathematics/I9,I9,Subtraction of 4 digit numbers,TIMSS 1995,4,Mathematics,Whole Numbers,NaN,Performing Routine Procedures,multiple_choice,1,False,71.0,71.0,"Subtract:\n\n 6,000\n−2,369\n____\n\nA. 4,369..."
3,TIMSS 1995/grade 4/Mathematics/J4,J4,What is the increase in product,TIMSS 1995,4,Mathematics,Whole Numbers,NaN,Using Complex Procedures,multiple_choice,1,False,45.0,46.0,25 × 18 is more than 24 × 18. How much more?\n...
4,TIMSS 1995/grade 4/Mathematics/J9,J9,Number in box,TIMSS 1995,4,Mathematics,Whole Numbers,NaN,Performing Routine Procedures,multiple_choice,1,True,73.0,83.0,Here is part of a wall chart that lists number...


### Embed the chunks

`gemini-embedding-001` embeds the text of each chunk as a 768-dimensional vector with the `RETRIEVAL_DOCUMENT` task type.
Vectors with fewer than the model's 3,072 dimensions are not unit length, so they are normalized, which makes cosine similarity a dot product.

`data/timss_math_item_embeddings.npz` stores the vectors in chunk order, with each item's key and the SHA-256 hash of its text.
A rerun reuses the stored vector of every unchanged text and requests only the others, so a changed item is re-embedded and an interrupted run resumes where it stopped.

The free tier counts every text in a batch request as one request and allows 100 a minute, so `embed_batch` waits as needed to send at most `EMBEDDING_TEXTS_PER_MINUTE` texts in any minute; embedding every item takes about 7 minutes.
A failed request is not retried, because the free tier also has daily quotas; rerun the cell once the quota allows.

Embed a search query as `RETRIEVAL_QUERY`.
Embed a new item like the stored ones (its `item_text`, as `RETRIEVAL_DOCUMENT`), so its similarities to them are comparable.

In [3]:
EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 768
DOCUMENT_TASK_TYPE = "RETRIEVAL_DOCUMENT"
EMBEDDING_BATCH_SIZE = 100  # the most texts one batchEmbedContents request takes
# The free-tier quota, in which every text of a batch counts as one request. The window has a second to spare for
# differences between this clock and the server's.
EMBEDDING_TEXTS_PER_MINUTE = 100
EMBEDDING_QUOTA_SECONDS = 61
EMBEDDINGS_PATH = DATA_DIR / "timss_math_item_embeddings.npz"

if not gemini_api_key:
    raise RuntimeError("Set GEMINI_API_KEY in the project .env before embedding.")
if "embedding_requests" not in globals():  # kept when the cell is rerun, so the quota still counts them
    embedding_requests = deque()  # (time.monotonic(), number of texts) of each recent request


def wait_for_embedding_quota(text_count):
    """Sleep until sending `text_count` more texts keeps the texts sent in the quota window within the quota."""
    if text_count > EMBEDDING_TEXTS_PER_MINUTE:
        raise ValueError(f"A request can embed at most {EMBEDDING_TEXTS_PER_MINUTE} texts")
    while True:
        now = time.monotonic()
        while embedding_requests and now - embedding_requests[0][0] >= EMBEDDING_QUOTA_SECONDS:
            embedding_requests.popleft()
        if sum(count for _, count in embedding_requests) + text_count <= EMBEDDING_TEXTS_PER_MINUTE:
            break
        time.sleep(EMBEDDING_QUOTA_SECONDS - (now - embedding_requests[0][0]))
    embedding_requests.append((time.monotonic(), text_count))


def embed_batch(texts, task_type):
    """Unit-length embeddings of `texts`, one row per text, from one batchEmbedContents request.

    A failed request raises instead of being retried, so retries do not use up the free-tier quota.
    """
    wait_for_embedding_quota(len(texts))
    response = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/{EMBEDDING_MODEL}:batchEmbedContents",
        headers={"x-goog-api-key": gemini_api_key},
        json={
            "requests": [
                {
                    "model": f"models/{EMBEDDING_MODEL}",
                    "content": {"parts": [{"text": text}]},
                    "taskType": task_type,
                    "outputDimensionality": EMBEDDING_DIMENSION,
                }
                for text in texts
            ]
        },
        timeout=120,
    )
    if not response.ok:
        raise RuntimeError(f"{EMBEDDING_MODEL} returned HTTP {response.status_code}: {response.text}")
    vectors = np.array([embedding["values"] for embedding in response.json()["embeddings"]], dtype=np.float32)
    if vectors.shape != (len(texts), EMBEDDING_DIMENSION):
        raise RuntimeError(f"Expected {len(texts)} vectors of {EMBEDDING_DIMENSION} dimensions, got {vectors.shape}")
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


@functools.cache
def embed_query(text, task_type="RETRIEVAL_QUERY"):
    """Embedding of one search text, cached so that repeating a search does not call the API again."""
    return embed_batch([text], task_type)[0].tolist()


def text_sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


chunk_hashes = [text_sha256(chunk["text"]) for chunk in chunks]
vectors_by_hash = {}
stored_rows = []  # (item key, text hash) of each stored vector
if EMBEDDINGS_PATH.exists():
    with np.load(EMBEDDINGS_PATH) as stored:
        settings = (str(stored["model"]), int(stored["dimension"]), str(stored["task_type"]))
        if settings == (EMBEDDING_MODEL, EMBEDDING_DIMENSION, DOCUMENT_TASK_TYPE):
            vectors_by_hash = dict(zip(stored["text_sha256"].tolist(), stored["embeddings"]))
            stored_rows = list(zip(stored["item_keys"].tolist(), stored["text_sha256"].tolist()))

texts_by_hash = dict(zip(chunk_hashes, (chunk["text"] for chunk in chunks)))
hashes_to_embed = [text_hash for text_hash in texts_by_hash if text_hash not in vectors_by_hash]
print(f"{len(texts_by_hash) - len(hashes_to_embed)} texts already embedded, {len(hashes_to_embed)} to embed")
try:
    for start in range(0, len(hashes_to_embed), EMBEDDING_BATCH_SIZE):
        batch = hashes_to_embed[start : start + EMBEDDING_BATCH_SIZE]
        vectors = embed_batch([texts_by_hash[text_hash] for text_hash in batch], DOCUMENT_TASK_TYPE)
        vectors_by_hash.update(zip(batch, vectors))
        print(f"Embedded {start + len(batch)}/{len(hashes_to_embed)} texts")
finally:
    # Store whatever is embedded, also when a request fails, so a rerun resumes from there. Vectors of items no
    # longer in the JSONL, or of their old text, are dropped.
    rows = [
        (chunk["item_key"], text_hash)
        for chunk, text_hash in zip(chunks, chunk_hashes)
        if text_hash in vectors_by_hash
    ]
    if rows != stored_rows:
        np.savez_compressed(
            EMBEDDINGS_PATH,
            item_keys=np.array([key for key, _ in rows]),
            text_sha256=np.array([text_hash for _, text_hash in rows]),
            embeddings=np.array([vectors_by_hash[text_hash] for _, text_hash in rows]).reshape(-1, EMBEDDING_DIMENSION),
            model=EMBEDDING_MODEL,
            dimension=EMBEDDING_DIMENSION,
            task_type=DOCUMENT_TASK_TYPE,
        )
        print(f"Stored {len(rows)} vectors in {EMBEDDINGS_PATH}")

item_embeddings = np.array([vectors_by_hash[text_hash] for text_hash in chunk_hashes])
print(f"Item embeddings: {item_embeddings.shape}")

728 texts already embedded, 0 to embed
Stored 728 vectors in /Users/ze/Documents/projects/timss-1/notebooks/data/timss_math_item_embeddings.npz
Item embeddings: (728, 768)


### Store the chunks in Qdrant

The local Qdrant collection `timss_math_items` (in `data/qdrant_timss_math_items/`) holds one point per chunk, with the chunk as its payload and two named vectors:

- `dense`, the Gemini embedding, compared by cosine similarity, for semantic retrieval.
- `bm25`, a sparse BM25 vector of the text, for keyword retrieval. Qdrant keeps sparse vectors in an inverted index and computes each term's inverse document frequency over the collection at query time (the IDF modifier), so the stored vector holds only the BM25 term-frequency part.

Keyword terms are lowercase words, folded to the singular by Harman's S stemmer, and numbers without thousands separators, so "1,000 marbles" matches "1000 marble".
The index of a term in a sparse vector is a 32-bit hash of the term, so no vocabulary has to be stored.

Rerunning the cell rebuilds the collection from the stored vectors.
The storage is separate from `data/qdrant/`, which the kernel of `get_data.ipynb` locks while its client is open.
This notebook's kernel likewise locks `data/qdrant_timss_math_items/` until `qdrant_client.close()`.
Local Qdrant filters by scanning the payloads, which is fast at this size; on a Qdrant server, add payload indexes for the fields you filter on.

In [4]:
QDRANT_PATH = DATA_DIR / "qdrant_timss_math_items"
QDRANT_COLLECTION = "timss_math_items"
DENSE_VECTOR = "dense"
SPARSE_VECTOR = "bm25"
BM25_K1 = 1.2  # term-frequency saturation
BM25_B = 0.75  # document-length normalization
TERM = re.compile(r"(?P<number>\d{1,3}(?:,\d{3})+(?!\d)(?:\.\d+)?|\d+(?:\.\d+)?)|(?P<word>[^\W\d_]+)")


def s_stem(word):
    """Harman's S stemmer, which folds English plurals to the singular: cards -> card, pennies -> penny.

    Words of up to three letters, such as "is", "bus", and the "s" of "Maria's", are kept as they are.
    """
    if len(word) <= 3:
        return word
    if word.endswith("ies") and not word.endswith(("eies", "aies")):
        return word[:-3] + "y"
    if word.endswith("es") and not word.endswith(("aes", "ees", "oes")):
        return word[:-1]
    if word.endswith("s") and not word.endswith(("us", "ss")):
        return word[:-1]
    return word


def terms(text):
    """Keyword terms of a text: lowercase words folded to the singular, and numbers without thousands separators."""
    return [
        match["number"].replace(",", "") if match["number"] else s_stem(match["word"])
        for match in TERM.finditer(text.lower())
    ]


def term_index(term):
    """Index of a term in sparse vectors: a stable 32-bit hash, so no vocabulary has to be stored."""
    return zlib.crc32(term.encode("utf-8"))


def bm25_document_vector(text, average_length):
    """BM25 term-frequency weights of a chunk's terms. Qdrant multiplies them by each term's IDF at query time."""
    counts = Counter(map(term_index, terms(text)))
    length_norm = BM25_K1 * (1 - BM25_B + BM25_B * counts.total() / average_length)
    return models.SparseVector(
        indices=list(counts),
        values=[count * (BM25_K1 + 1) / (count + length_norm) for count in counts.values()],
    )


def bm25_query_vector(text):
    """Weight 1 for each distinct term of a query, so an item scores the sum of its BM25 weights of those terms."""
    indices = sorted(set(map(term_index, terms(text))))
    return models.SparseVector(indices=indices, values=[1.0] * len(indices))


def point_id(key):
    """Qdrant point ID of the item with this item_key, the same on every rebuild."""
    return str(uuid5(NAMESPACE_URL, key))


bm25_average_length = float(np.mean([len(terms(chunk["text"])) for chunk in chunks]))

if "qdrant_client" in globals():
    qdrant_client.close()
qdrant_client = QdrantClient(path=str(QDRANT_PATH))
if qdrant_client.collection_exists(QDRANT_COLLECTION):
    qdrant_client.delete_collection(QDRANT_COLLECTION)
qdrant_client.create_collection(
    QDRANT_COLLECTION,
    vectors_config={DENSE_VECTOR: models.VectorParams(size=EMBEDDING_DIMENSION, distance=models.Distance.COSINE)},
    sparse_vectors_config={SPARSE_VECTOR: models.SparseVectorParams(modifier=models.Modifier.IDF)},
)
qdrant_client.upload_points(
    QDRANT_COLLECTION,
    [
        models.PointStruct(
            id=point_id(chunk["item_key"]),
            vector={
                DENSE_VECTOR: vector.tolist(),
                SPARSE_VECTOR: bm25_document_vector(chunk["text"], bm25_average_length),
            },
            payload=chunk,
        )
        for chunk, vector in zip(chunks, item_embeddings)
    ],
    wait=True,
)
point_count = qdrant_client.count(QDRANT_COLLECTION, exact=True).count
if point_count != len(chunks):
    raise RuntimeError(f"{QDRANT_COLLECTION} holds {point_count} points instead of {len(chunks)}")
print(f"Stored {point_count} items in collection '{QDRANT_COLLECTION}' at {QDRANT_PATH}")

Stored 728 items in collection 'timss_math_items' at /Users/ze/Documents/projects/timss-1/notebooks/data/qdrant_timss_math_items


### Search the collection

`search(text, method)` returns the items that best match `text`, best first:

- `"keyword"` ranks items by the BM25 score of their stem and options.
- `"semantic"` ranks them by the cosine similarity of their embedding with the embedding of `text`.
- `"hybrid"` (the default) fuses the keyword and semantic rankings, each of up to `HYBRID_CANDIDATES` items, by reciprocal rank fusion, so items ranked high by either method or by both come first. Its scores are fusion scores, not similarities.

`query_filter` keeps any method to items with the given metadata; `metadata_filter` builds one from field values, where a list accepts any of its values.
`text` is embedded as `RETRIEVAL_QUERY`, for questions and keywords.
To find the items most similar to an item, pass its `item_text` with `task_type=DOCUMENT_TASK_TYPE`, as the stored items were embedded.
`vector` passes the embedding of `text` when it is already known, such as a stored item's, so it is not requested again.

In [5]:
HYBRID_CANDIDATES = 50  # items each method contributes to the fusion
PAYLOAD_FIELDS = set(chunks[0])


def metadata_filter(**conditions):
    """Qdrant filter that keeps the items matching every condition: field=value, or field=[values] for any of them.

    None leaves a field unfiltered. For example, metadata_filter(assessment=["TIMSS 2007", "TIMSS 2011"], grade=4).
    """
    unknown_fields = set(conditions) - PAYLOAD_FIELDS
    if unknown_fields:
        raise ValueError(f"Unknown fields {sorted(unknown_fields)}; the payload fields are {sorted(PAYLOAD_FIELDS)}")
    return models.Filter(
        must=[
            models.FieldCondition(
                key=field,
                match=models.MatchAny(any=value) if isinstance(value, list) else models.MatchValue(value=value),
            )
            for field, value in conditions.items()
            if value is not None
        ]
    )


def search(text, method="hybrid", limit=10, query_filter=None, task_type="RETRIEVAL_QUERY", vector=None):
    """Items that best match `text`, best first, by "keyword" (BM25), "semantic" (embeddings), or "hybrid" (both).

    `vector` is the embedding of `text` if it is already known, so the semantic methods do not request it again.
    """
    queries = {}
    if method in ("keyword", "hybrid"):
        queries[SPARSE_VECTOR] = bm25_query_vector(text)
    if method in ("semantic", "hybrid"):
        queries[DENSE_VECTOR] = embed_query(text, task_type) if vector is None else list(map(float, vector))
    if not queries:
        raise ValueError(f"method must be 'keyword', 'semantic', or 'hybrid', not {method!r}")
    if method == "hybrid":
        return qdrant_client.query_points(
            QDRANT_COLLECTION,
            prefetch=[
                models.Prefetch(query=query, using=using, filter=query_filter, limit=HYBRID_CANDIDATES)
                for using, query in queries.items()
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=limit,
        ).points
    [(using, query)] = queries.items()
    return qdrant_client.query_points(
        QDRANT_COLLECTION, query=query, using=using, query_filter=query_filter, limit=limit
    ).points


def results_table(points):
    """One row per search result: its score, item, and the start of its text."""
    return pd.DataFrame(
        [
            {
                "score": point.score,
                **{field: point.payload[field] for field in ("assessment", "grade", "item_id", "content_domain")},
                "text": " ".join(point.payload["text"].split())[:100],
            }
            for point in points
        ]
    )

In [6]:
query = "fraction of a shape that is shaded"
for method in ("keyword", "semantic", "hybrid"):
    print(method)
    display(results_table(search(query, method, limit=5)))

print("hybrid, grade 4 items of TIMSS 2007 and 2011")
display(
    results_table(
        search(query, limit=5, query_filter=metadata_filter(assessment=["TIMSS 2007", "TIMSS 2011"], grade=4))
    )
)

keyword


,score,assessment,grade,item_id,content_domain,text
0,11.020040,TIMSS 1995,4,J7,Fractions and Proportionality,Part of the figure is shaded. [Image: A 3x3 gr...
1,10.896976,TIMSS 2003,4,M031347C,Number,C. What fraction of the figure is shaded in pa...
2,10.777558,TIMSS 2007,4,M041006,Number,What fraction of this rectangle is shaded? [Im...
3,9.750047,TIMSS 2007,8,M022043,Number,[Image: Rectangle divided into 12 squares with...
4,9.490537,TIMSS 2007,4,M041173,Geometric Shapes and Measures,[Image: A shaded geometric shape] The shape ab...


semantic


,score,assessment,grade,item_id,content_domain,text
0,0.778227,TIMSS 1995,4,J7,Fractions and Proportionality,Part of the figure is shaded. [Image: A 3x3 gr...
1,0.775646,TIMSS 2007,4,M041006,Number,What fraction of this rectangle is shaded? [Im...
2,0.763921,TIMSS 1999,8,F12,Fractions and Number Sense,What fraction of the circle is shaded? [Image:...
3,0.763829,TIMSS 2003,4,M031347C,Number,C. What fraction of the figure is shaded in pa...
4,0.761252,TIMSS 2003,4,M012044,Number,Which shows 2/3 of the square shaded? A. [Imag...


hybrid


,score,assessment,grade,item_id,content_domain,text
0,1.000000,TIMSS 1995,4,J7,Fractions and Proportionality,Part of the figure is shaded. [Image: A 3x3 gr...
1,0.583333,TIMSS 2007,4,M041006,Number,What fraction of this rectangle is shaded? [Im...
2,0.533333,TIMSS 2003,4,M031347C,Number,C. What fraction of the figure is shaded in pa...
3,0.392857,TIMSS 1999,8,F12,Fractions and Number Sense,What fraction of the circle is shaded? [Image:...
4,0.266667,TIMSS 2007,8,M022043,Number,[Image: Rectangle divided into 12 squares with...


hybrid, grade 4 items of TIMSS 2007 and 2011


,score,assessment,grade,item_id,content_domain,text
0,1.000000,TIMSS 2007,4,M041006,Number,What fraction of this rectangle is shaded? [Im...
1,0.476190,TIMSS 2007,4,M041173,Geometric Shapes and Measures,[Image: A shaded geometric shape] The shape ab...
2,0.375000,TIMSS 2011,4,M031297,Geometric Shapes and Measures,[Image: a shaded area overlaying a grid made u...
3,0.333333,TIMSS 2011,4,M041284,Geometric Shapes and Measures,"[Image: various shapes labeled A, B, C, D, E, ..."
4,0.333333,TIMSS 2011,4,M041064,Number,Shade 1/2 of the large triangle. [Image: a lar...


In [7]:
# The items most similar to one released item, the item itself left out. The other parts of a multi-part item, and the
# other items of its task (M031379 and M031380 here), share its stimulus, so they rank first.
example_item = next(item for item in released_items if item["item_id"] == "M031346A")
display(
    results_table(
        search(
            item_text(example_item),
            limit=5,
            query_filter=models.Filter(must_not=[models.HasIdCondition(has_id=[point_id(item_key(example_item))])]),
            task_type=DOCUMENT_TASK_TYPE,
        )
    )
)

,score,assessment,grade,item_id,content_domain,text
0,1.000000,TIMSS 2011,4,M031346C,Number,The town fair had a booth where people could t...
1,0.666667,TIMSS 2011,4,M031346B,Number,The town fair had a booth where people could t...
2,0.450000,TIMSS 2011,4,M031379,Number,Trading Cards (Continued) Trading Sports Cards...
3,0.450000,TIMSS 2011,4,M031380,Number,Trading Cartoon Cards Brad had 8 cartoon cards...
4,0.233333,TIMSS 2007,4,M031247,Number,A man took his 3 children to a fair. Tickets c...


## Add the item parameters to the stored metadata

`data/timss_math_released_item_parameters.xlsx` (sheet `item_parameters`, written by `get_data.ipynb` and documented in `timss_math_released_items.md`) has the official IRT parameters of the released items.
Each item in the collection gets its parameters as an `item_parameters` payload object, matched by `item_key`:

- `irt_model`: 3PL, 2PL, or GPCM.
- `scaled_points`: the item's points in the IRT scaling, 1 for a 3PL or 2PL item and the number of steps for a GPCM item.
- `slope`, `location`, `guessing`, `step1`, `step2`, and `step3`: the estimates, null for parameters the model does not have.
- `slope_se` and so on: the standard error of each estimate.
- `calibration`: the concurrent calibration the parameters come from.

The cell checks that every item in the collection has a row in the workbook, that each item's model fits its format (3PL for multiple choice, 2PL for a constructed-response item worth 1 point in the scaling, GPCM with one step per point for one worth more), and that the steps of every GPCM item sum to 0.
K14 and L18 of TIMSS Advanced 1995 are worth 2 points in the 1995 coding guide but were scaled as 1-point items, so they are 2PL items with `scaled_points` 1.

Six items in the collection have no parameters, so their `item_parameters` is null: K8, K9, L10, L11, and L15A of TIMSS Advanced 1995, which the 1995-2008 rescaling left out, and M042273 of TIMSS 2007 grade 8, which was not used in scaling.
They stay in the collection for search but are never reference items, and they are not predicted.

Within a grade, the parameters come from different concurrent calibrations.
At grade 4, the TIMSS 1995 and 2003 items come from the joint 1995-2003 calibration, the TIMSS 2007 items from the joint 2003-2007 calibration, and the TIMSS 2011 items from the 2007-2011 calibration; grade 8 is the same with 1999 in place of 1995.
TIMSS links each calibration to the one before through trend items, so their scales are close but not identical.
TIMSS Advanced 1995 and 2008 share one calibration.

`calibrated_table` holds the parameters of the items in the collection that have them, one row per item, for the prediction code below.
Rerunning "Store the chunks in Qdrant" rebuilds the collection without parameters, so rerun this cell after it.

In [8]:
PARAMETER_NAMES = ["slope", "location", "guessing", "step1", "step2", "step3"]
STEP_NAMES = ["step1", "step2", "step3"]
ITEM_PARAMETERS_PAYLOAD = "item_parameters"
# Matches the items without parameters: Qdrant's is_empty condition matches a null payload field.
NO_PARAMETERS = models.IsEmptyCondition(is_empty=models.PayloadField(key=ITEM_PARAMETERS_PAYLOAD))

parameter_table = pd.read_excel(ITEM_PARAMETERS_PATH, sheet_name="item_parameters")
parameter_table.index = parameter_table.apply(item_key, axis=1).rename("item_key")
if not parameter_table.index.is_unique:
    raise ValueError(f"Item keys repeat in {ITEM_PARAMETERS_PATH.name}")
missing_rows = sorted(set(chunks_table["item_key"]) - set(parameter_table.index))
if missing_rows:
    raise ValueError(f"No row in {ITEM_PARAMETERS_PATH.name} for {missing_rows}")

calibrated_table = parameter_table.loc[chunks_table["item_key"]]
calibrated_table = calibrated_table[calibrated_table["irt_model"].notna()].copy()
calibrated_table["scaled_points"] = np.where(
    calibrated_table["irt_model"] == "GPCM", calibrated_table[STEP_NAMES].notna().sum(axis=1), 1
)

# The model must fit the format and points in the scaling, as in the TIMSS scaling.
expected_model = np.where(
    calibrated_table["item_type"] == "multiple_choice",
    "3PL",
    np.where(calibrated_table["scaled_points"] == 1, "2PL", "GPCM"),
)
misfits = calibrated_table.index[calibrated_table["irt_model"] != expected_model].tolist()
if misfits:
    raise ValueError(f"The IRT model does not fit the item format of {misfits}")
for model_name, parameters in {"3PL": ["slope", "location", "guessing"], "2PL": ["slope", "location"]}.items():
    rows = calibrated_table[calibrated_table["irt_model"] == model_name]
    others = [name for name in PARAMETER_NAMES if name not in parameters]
    if rows[parameters].isna().any().any() or rows[others].notna().any().any():
        raise ValueError(f"{model_name} items must have exactly the parameters {parameters}")
gpcm_rows = calibrated_table[calibrated_table["irt_model"] == "GPCM"]
if gpcm_rows[["slope", "location", "step1", "step2"]].isna().any().any() or gpcm_rows["guessing"].notna().any():
    raise ValueError("GPCM items need a slope, a location, and at least two steps, and no guessing parameter")
unbalanced_steps = gpcm_rows.index[gpcm_rows[STEP_NAMES].sum(axis=1).abs() > 0.002].tolist()
if unbalanced_steps:
    raise ValueError(f"The steps of {unbalanced_steps} do not sum to 0")
if calibrated_table.groupby(["assessment", "grade"])["calibration"].nunique().gt(1).any():
    raise ValueError("Every assessment and grade should have one calibration")


def parameter_payload(key):
    """The item_parameters payload of the item with this key, or None if it has no parameters."""
    if key not in calibrated_table.index:
        return None
    row = calibrated_table.loc[key]
    number = lambda value: None if pd.isna(value) else float(value)
    return {
        "irt_model": row["irt_model"],
        "scaled_points": int(row["scaled_points"]),
        **{name: number(row[name]) for name in PARAMETER_NAMES},
        **{f"{name}_se": number(row[f"{name}_se"]) for name in PARAMETER_NAMES},
        "calibration": row["calibration"],
    }


qdrant_client.batch_update_points(
    QDRANT_COLLECTION,
    [
        models.SetPayloadOperation(
            set_payload=models.SetPayload(payload={ITEM_PARAMETERS_PAYLOAD: parameter_payload(key)}, points=[point_id(key)])
        )
        for key in chunks_table["item_key"]
    ],
    wait=True,
)
stored_with_parameters = qdrant_client.count(
    QDRANT_COLLECTION, count_filter=models.Filter(must_not=[NO_PARAMETERS]), exact=True
).count
if stored_with_parameters != len(calibrated_table):
    raise RuntimeError(f"{stored_with_parameters} points have parameters instead of {len(calibrated_table)}")

print(f"Added parameters to {stored_with_parameters} of {len(chunks_table)} items in '{QDRANT_COLLECTION}'")
display(pd.crosstab([calibrated_table["grade"], calibrated_table["assessment"]], calibrated_table["irt_model"], margins=True))
print("Items without parameters:")
display(
    chunks_table.loc[~chunks_table["item_key"].isin(calibrated_table.index), ["assessment", "grade", "item_id"]]
    .reset_index(drop=True)
)
scaled_differently = calibrated_table[calibrated_table["scaled_points"] != calibrated_table["max_points"]]
print("Items whose points in the scaling differ from their maximum points:")
display(
    scaled_differently[["assessment", "item_id", "item_type", "max_points", "scaled_points", "irt_model"]].reset_index(
        drop=True
    )
)

Added parameters to 722 of 728 items in 'timss_math_items'


irt_model                  2PL  3PL  GPCM  All
grade assessment                              
4     TIMSS 1995            22   41     6   69
      TIMSS 2003            25   51     3   79
      TIMSS 2007            35   36     3   74
      TIMSS 2011            31   36     6   73
8     TIMSS 1999            14   64     3   81
      TIMSS 2003            21   72     6   99
      TIMSS 2007            27   50    11   88
      TIMSS 2011            30   48    10   88
12    TIMSS Advanced 1995    6   19     6   31
      TIMSS Advanced 2008   13   25     2   40
All                        224  442    56  722

Items without parameters:


,assessment,grade,item_id
0,TIMSS Advanced 1995,12,K8
1,TIMSS Advanced 1995,12,K9
2,TIMSS Advanced 1995,12,L10
3,TIMSS Advanced 1995,12,L11
4,TIMSS Advanced 1995,12,L15A
5,TIMSS 2007,8,M042273


Items whose points in the scaling differ from their maximum points:


,assessment,item_id,item_type,max_points,scaled_points,irt_model
0,TIMSS Advanced 1995,K14,constructed_response,2,1,2PL
1,TIMSS Advanced 1995,L18,constructed_response,2,1,2PL


## Predict the parameters of a new item

`predict_item_parameters(item)` predicts the IRT parameters of a new item in two steps.

1. `find_similar_items` returns the calibrated items of the collection most similar to the new item, only from the item's grade.
   It searches with the item's text (stem and options) by `SIMILAR_ITEMS_METHOD`, hybrid search by default, so items that share the new item's words or meaning rank first.
   The new item is embedded like the stored items (`RETRIEVAL_DOCUMENT`), so its cosine similarities with them are comparable; an item whose text is in the collection reuses its stored vector instead of calling the embedding API.
   Items without parameters, and the items in `exclude_keys`, are never returned.
2. An LLM (`PARAMETER_LLM_MODEL`, through OpenRouter) gets the similar items as context and returns the new item's parameters and its reasoning in a structured-output schema.

The context holds the `N_SIMILAR` most similar items of any format, then the `N_SAME_MODEL` most similar items scored with the new item's IRT model and points that are not among them, so the LLM always sees calibrated examples of every parameter it predicts.
For each of these reference items, the LLM gets the whole item (classification, format, stem, the model-written figure description, options and correct option or scoring guide), its cosine similarity with the new item, its international average percent correct, its calibration, and its calibrated parameters.
It also gets a calibration summary: the distribution of each parameter over all calibrated items of the grade scored with the same model and points, without the new item.
The new item's own percent correct is never given, since a new item has none.

The IRT model follows the item's format and points, as in the TIMSS scaling:

| Item | IRT model | Parameters | The LLM predicts |
| --- | --- | --- | --- |
| Multiple choice | 3PL | slope (a), location (b), guessing (c) | all three |
| Constructed response worth 1 point (scored 0 or 1) | 2PL | slope, location | both |
| Constructed response worth m > 1 points (partial credit) | GPCM | slope, location, steps d_1 to d_m | slope, location, d_1 to d_(m-1) |

The steps of every calibrated GPCM item sum to 0, so the last step is minus the sum of the others; for a 2-point item, step 2 is minus step 1.

The system prompt (`parameter_system_prompt`) makes the LLM an expert in testing the subject for the item's grade who is very familiar with the item's content domain (and main topic, where the framework has one) and cognitive domain, and a measurement expert in IRT models.
It states the three models with the TIMSS scaling constant 1.7, explains each parameter, says the reference items come from linked but not identical calibrations, and asks the LLM to first compare the new item with the reference items and then give its reasoning for each parameter before the value.

A new item is a `TimssItem`, with the fields of the released-items JSONL that describe it.
Its `assessment` names the calibration whose scale the prediction targets (`CALIBRATIONS`); for an assessment without calibrated items, the prompt asks for the scale of the most recent calibration among the reference items.
`prepare_prediction` finds the reference items and writes the prompts without calling the LLM, `request_prediction` sends the prompts, and `predict_item_parameters` does both.

In [9]:
SIMILAR_ITEMS_METHOD = "hybrid"  # "keyword", "semantic", or "hybrid"; see search
N_SIMILAR = 10  # most similar items of any format
N_SAME_MODEL = 5  # most similar items scored with the new item's IRT model and points, if not among the N_SIMILAR
PARAMETER_LABELS = {
    "slope": "slope (a)",
    "location": "location (b)",
    "guessing": "guessing (c)",
    "step1": "step 1 (d_1)",
    "step2": "step 2 (d_2)",
    "step3": "step 3 (d_3)",
}
# The calibration of each assessment and grade: the scale that a prediction for an item of it targets.
CALIBRATIONS = calibrated_table.groupby(["assessment", "grade"])["calibration"].first().to_dict()


class ItemOption(BaseModel):
    label: str
    text: str


class ScoringCategory(BaseModel):
    category: str
    descriptors: list[str]


class TimssItem(BaseModel):
    """An item whose parameters are predicted, with the fields of the released-items JSONL that describe it.

    `assessment` names the calibration the prediction targets (see CALIBRATIONS). `scaled_points` is the item's points
    in the IRT scaling when they differ from `max_points`, as for K14 and L18 of TIMSS Advanced 1995.
    """

    item_id: str | None = None
    assessment: str
    grade: Literal[4, 8, 12]
    subject: str
    content_domain: str
    main_topic: str | None = None
    cognitive_domain: str
    item_type: Literal["multiple_choice", "constructed_response"]
    max_points: int = Field(default=1, ge=1)
    scaled_points: int | None = Field(default=None, ge=1)
    stem: str
    options: list[ItemOption] | None = None
    correct_option: str | None = None
    scoring_guide: list[ScoringCategory] | None = None
    scoring_notes: str | None = None
    figure_description: str | None = None

    @model_validator(mode="after")
    def check_format(self):
        if self.item_type == "multiple_choice":
            labels = [option.label for option in self.options or []]
            if len(labels) < 2 or self.correct_option not in labels or self.points != 1:
                raise ValueError("A multiple-choice item needs options, a correct option among them, and 1 point")
        elif self.options or self.correct_option:
            raise ValueError("A constructed-response item has no options or correct option")
        return self

    @property
    def points(self):
        """The item's points in the IRT scaling."""
        return self.scaled_points or self.max_points

    @property
    def irt_model(self):
        """3PL for multiple choice, 2PL for a 1-point and GPCM for a polytomous constructed-response item."""
        if self.item_type == "multiple_choice":
            return "3PL"
        return "2PL" if self.points == 1 else "GPCM"


def model_parameters(irt_model, points):
    """The parameters of an item scored with `irt_model` and worth `points` in the scaling."""
    if irt_model == "3PL":
        return ["slope", "location", "guessing"]
    if irt_model == "2PL":
        return ["slope", "location"]
    return ["slope", "location", *[f"step{step}" for step in range(1, points + 1)]]


def predicted_parameters(irt_model, points):
    """The parameters the LLM predicts: all but the last step of a GPCM item, which is minus the sum of the others."""
    names = model_parameters(irt_model, points)
    return names[:-1] if irt_model == "GPCM" else names


def item_vector(item):
    """Embedding of a TimssItem's text: the stored vector if its text is in the collection, else a new embedding."""
    text = item_text(item.model_dump())
    stored = vectors_by_hash.get(text_sha256(text))
    return np.asarray(embed_query(text, DOCUMENT_TASK_TYPE) if stored is None else stored)


def find_similar_items(item, limit=10, irt_model=None, scaled_points=None, exclude_keys=(), method=SIMILAR_ITEMS_METHOD):
    """The calibrated items of the collection most similar to `item`, most similar first, all of the item's grade.

    `irt_model` and `scaled_points` keep only items scored with that model and worth that many points in the scaling.
    `exclude_keys` leaves items out, such as the item itself and its other parts. Returns one dict per item with its
    rank, item_key, cosine similarity with `item`, and payload.
    """
    must = [models.FieldCondition(key="grade", match=models.MatchValue(value=item.grade))]
    if irt_model is not None:
        must.append(
            models.FieldCondition(key=f"{ITEM_PARAMETERS_PAYLOAD}.irt_model", match=models.MatchValue(value=irt_model))
        )
    if scaled_points is not None:
        must.append(
            models.FieldCondition(
                key=f"{ITEM_PARAMETERS_PAYLOAD}.scaled_points", match=models.MatchValue(value=scaled_points)
            )
        )
    must_not = [NO_PARAMETERS]
    if exclude_keys:
        must_not.append(models.HasIdCondition(has_id=[point_id(key) for key in exclude_keys]))
    vector = item_vector(item)
    points = search(
        item_text(item.model_dump()),
        method,
        limit,
        models.Filter(must=must, must_not=must_not),
        task_type=DOCUMENT_TASK_TYPE,
        vector=vector,
    )
    return [
        {
            "rank": rank,
            "item_key": point.payload["item_key"],
            "cosine_similarity": float(vector @ vectors_by_hash[text_sha256(point.payload["text"])]),
            "payload": point.payload,
        }
        for rank, point in enumerate(points, 1)
    ]


def similar_items_table(similar_items):
    """One row per item from find_similar_items: rank, similarity, classification, and calibrated parameters."""
    return pd.DataFrame(
        [
            {
                "rank": entry["rank"],
                "cosine_similarity": entry["cosine_similarity"],
                **{
                    field: entry["payload"][field]
                    for field in ("assessment", "item_id", "content_domain", "cognitive_domain", "item_type")
                },
                "pct_correct_intl_avg": entry["payload"]["pct_correct_intl_avg"],
                **{
                    field: entry["payload"][ITEM_PARAMETERS_PAYLOAD][field]
                    for field in ("irt_model", "scaled_points", *PARAMETER_NAMES)
                },
            }
            for entry in similar_items
        ]
    )

In [10]:
STUDENTS = {
    4: "fourth-grade students",
    8: "eighth-grade students",
    12: "students in their final year of secondary school who have taken advanced mathematics courses (TIMSS Advanced)",
}


def plural(count, noun):
    return f"{count} {noun}{'' if count == 1 else 's'}"


def parameter_system_prompt(item):
    """System prompt: an expert in testing the subject for the item's grade, content domain, and cognitive domain, and
    a measurement expert in IRT models."""
    subject = item.subject.lower()
    students = STUDENTS[item.grade]
    topic = f", especially {item.main_topic}," if item.main_topic else ""
    return f"""\
You are an expert in testing {subject} for {students}. You are very familiar with assessing these students in the \
{item.content_domain} content domain{topic} and in the {item.cognitive_domain} cognitive domain, as the \
{item.assessment} framework defines them. You have written, reviewed, and analyzed many {subject} items for these \
students, and you know from experience how students across the TIMSS countries perform on them.

You are also a measurement expert in item response theory (IRT). You know the three-parameter logistic (3PL), \
two-parameter logistic (2PL), and generalized partial credit (GPCM) models, how TIMSS calibrates items with them, and \
how content, cognitive demand, the number and difficulty of steps, the numbers involved, reading load, figures, \
response format, distractor plausibility, and scoring rules shape item parameters.

Your task is to predict the IRT parameters that a new item would receive in a TIMSS calibration, using calibrated \
reference items of the same grade.

TIMSS scales items with these models, where theta is student ability and 1.7 is the scaling constant:
- 3PL, for multiple-choice items: P(correct | theta) = c + (1 - c) / (1 + exp(-1.7 a (theta - b))).
- 2PL, for constructed-response items scored 0 or 1: P(correct | theta) = 1 / (1 + exp(-1.7 a (theta - b))).
- GPCM, for constructed-response items scored from 0 to m points with partial credit: P(score = k | theta) is \
proportional to exp(sum over v = 0..k of 1.7 a (theta - b + d_v)), with d_0 = 0 and d_1 + ... + d_m = 0.

The parameters:
- Slope (a): discrimination, how sharply the chance of success rises with ability. Always positive.
- Location (b): overall difficulty on the theta scale. Higher is harder.
- Guessing (c), 3PL only: the chance that a student of very low ability answers correctly. It is usually below 1 \
divided by the number of options, because plausible distractors attract low-ability students.
- Steps (d_1 to d_m), GPCM only: scores k - 1 and k are equally likely at theta = b - d_k. A positive d_1 means the \
first point is reached well below b. When these boundaries are out of order, a middle score is rarely the most \
likely one. The steps sum to 0, so give all steps but the last, which is minus the sum of the others.

The reference items come from the concurrent calibrations of the TIMSS assessments of this grade. TIMSS links each \
calibration to the one before through trend items, so their scales are close but not identical. Each reference item \
names its calibration; when the evidence is otherwise equal, give more weight to reference items from the calibration \
that the new item is predicted for.

How to work:
1. Study the reference items, chosen for their similarity to the new item. Their percent correct shows how hard each \
one was for the students of its assessment, and their calibrated parameters anchor the scale.
2. Compare the new item with the most relevant reference items on content, cognitive demand, the number and difficulty \
of steps, the numbers involved, reading load, and figures, and also on distractor plausibility for a multiple-choice \
item or on how demanding the scoring guide is for a constructed-response item. Decide whether the new item should be \
easier or harder, and more or less discriminating, than each of them.
3. Treat the calibration summary for the new item's model as a prior, and stay within its range unless the new item \
is clearly unusual.
4. Combine the evidence instead of copying the parameters of one reference item.
5. First state which reference items are most comparable and how the new item differs from them. Then, for each \
parameter, give your reasoning before its value."""


def describe_item(item):
    """Plain-text rendering of a TimssItem, the same for reference items and the new item."""
    topic = f"; main topic: {item.main_topic}" if item.main_topic else ""
    lines = [f"Content domain: {item.content_domain}{topic}; cognitive domain: {item.cognitive_domain}"]
    if item.item_type == "multiple_choice":
        lines.append(f"Format: multiple choice with {len(item.options)} options, 1 point, scored with the 3PL model")
    else:
        points = plural(item.max_points, "point")
        if item.points != item.max_points:
            points += f" in the scoring guide, scaled as {plural(item.points, 'point')}"
        lines.append(f"Format: constructed response, {points}, scored with the {item.irt_model} model")
    lines.append(f"Stem:\n{item.stem}")
    if item.figure_description:
        lines.append(f"Figure, as described by a model (may be inexact):\n{item.figure_description}")
    if item.options:
        lines.append("Options:\n" + "\n".join(f"{option.label}. {option.text}" for option in item.options))
        lines.append(f"Correct option: {item.correct_option}")
    if item.scoring_guide:
        lines.append(
            "Scoring guide:\n"
            + "\n".join(f"- {category.category}: {' | '.join(category.descriptors)}" for category in item.scoring_guide)
        )
    if item.scoring_notes:
        lines.append(f"Scoring notes: {item.scoring_notes}")
    return "\n".join(lines)


def same_scoring_items(item, exclude_keys=()):
    """Calibrated items of the item's grade scored with its IRT model and points, except `exclude_keys`."""
    return calibrated_table[
        (calibrated_table["grade"] == item.grade)
        & (calibrated_table["irt_model"] == item.irt_model)
        & (calibrated_table["scaled_points"] == item.points)
        & ~calibrated_table.index.isin(list(exclude_keys))
    ]


def calibration_summary(item, exclude_keys=()):
    """The distribution of each parameter over same_scoring_items, the prior in the prompt."""
    scoring = f"the {item.irt_model} model" + (f" and {plural(item.points, 'point')}" if item.irt_model == "GPCM" else "")
    calibrated = same_scoring_items(item, exclude_keys)
    if calibrated.empty:
        return f"Calibration summary: no calibrated items of this grade are scored with {scoring}."
    summary = (
        calibrated[model_parameters(item.irt_model, item.points)]
        .rename(columns=PARAMETER_LABELS)
        .describe(percentiles=[0.1, 0.5, 0.9])
        .drop(index="count")
        .round(3)
    )
    return (
        f"Calibration summary: the parameters of the {len(calibrated)} calibrated items of this grade, from all its "
        f"assessments, that are scored with {scoring} (the new item is not among them):\n{summary.to_string()}"
    )


def format_parameters(parameters):
    """Calibrated parameters from an item_parameters payload, such as "slope (a) = 0.927, location (b) = 0.111"."""
    names = model_parameters(parameters["irt_model"], parameters["scaled_points"])
    return ", ".join(f"{PARAMETER_LABELS[name]} = {parameters[name]:.3f}" for name in names)


def target_scale(item):
    """The calibration whose scale the prediction targets, as the end of a sentence."""
    calibration = CALIBRATIONS.get((item.assessment, item.grade))
    if calibration is None:
        return "on the scale of the most recent calibration among the reference items"
    return f"that it would receive in the {calibration[0].lower()}{calibration[1:].rstrip('.')}"


def parameter_user_prompt(item, references, exclude_keys=()):
    """User prompt: the calibration summary, the reference items with their parameters, and the new item."""
    blocks = [calibration_summary(item, exclude_keys), "Reference items, most similar first:"]
    for reference in references:
        payload = reference["payload"]
        parameters = payload[ITEM_PARAMETERS_PAYLOAD]
        reference_item = TimssItem.model_validate(
            items_by_key[reference["item_key"]] | {"scaled_points": parameters["scaled_points"]}
        )
        pct_correct = payload["pct_correct_intl_avg"]
        blocks.append(
            f"### Reference item {reference['rank']}: {payload['item_id']} ({payload['assessment']}, "
            f"grade {payload['grade']})\n"
            f"Cosine similarity with the new item: {reference['cosine_similarity']:.3f}\n"
            f"{describe_item(reference_item)}\n"
            f"Percent correct, international average: {'not reported' if pct_correct is None else pct_correct}\n"
            f"Calibration: {parameters['calibration']}\n"
            f"Calibrated parameters: {format_parameters(parameters)}"
        )
    names = predicted_parameters(item.irt_model, item.points)
    request = f"Predict the parameters of the new item {target_scale(item)}. "
    request += f"Give its {', '.join(PARAMETER_LABELS[name] for name in names)}."
    if item.irt_model == "GPCM":
        request += f" Its {PARAMETER_LABELS[f'step{item.points}']} is minus the sum of the other steps."
    blocks.append(f"### New item ({item.assessment}, grade {item.grade})\n{describe_item(item)}\n\n{request}")
    return "\n\n".join(blocks)


@functools.cache
def prediction_schema(irt_model, points):
    """Structured-output schema of a prediction: the comparison with the reference items, then each parameter the LLM
    predicts, with its reasoning first."""
    fields = {
        "reasoning": (
            str,
            Field(description="The reference items most comparable to the new item, and how the new item differs from them."),
        )
    }
    for name in predicted_parameters(irt_model, points):
        fields[f"{name}_reasoning"] = (str, Field(description=f"Why the new item gets this {PARAMETER_LABELS[name]}."))
        if name == "slope":
            fields[name] = (float, Field(ge=0, description="Slope (a), positive."))
        elif name == "guessing":
            fields[name] = (float, Field(ge=0, le=1, description="Guessing (c), from 0 to 1."))
        else:
            fields[name] = (float, Field(description=f"{PARAMETER_LABELS[name].capitalize()}."))
    return create_model(f"{irt_model}Prediction{points}Points", **fields)

In [11]:
PARAMETER_LLM_MODEL = "google/gemini-3.8-flash"  # through OpenRouter
PARAMETER_REASONING_EFFORT = "medium"  # OpenRouter reasoning effort: "low", "medium", or "high"
PARAMETER_MAX_TOKENS = 16384  # output cap, reasoning included

if not openrouter_api_key:
    raise RuntimeError("Set OPENROUTER_API_KEY in the project .env before predicting.")
llm_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key, max_retries=2, timeout=600)


class NoPredictionError(RuntimeError):
    """The LLM returned no parsable prediction."""


def prepare_prediction(item, exclude_keys=()):
    """Reference items and prompts for predicting `item`, without calling the LLM.

    The reference items are the N_SIMILAR calibrated items of the item's grade most similar to it, then the
    N_SAME_MODEL most similar ones scored with its IRT model and points that are not among them. `exclude_keys` (such
    as the item and its other parts) are left out of the reference items and the calibration summary.
    """
    nearest = find_similar_items(item, N_SIMILAR, exclude_keys=exclude_keys)
    nearest_keys = {entry["item_key"] for entry in nearest}
    same_scoring = [
        entry
        for entry in find_similar_items(
            item, N_SAME_MODEL, irt_model=item.irt_model, scaled_points=item.points, exclude_keys=exclude_keys
        )
        if entry["item_key"] not in nearest_keys
    ]
    references = [
        entry | {"rank": rank, "found_by": found_by}
        for rank, (found_by, entry) in enumerate(
            [*(("most similar", entry) for entry in nearest), *(("same model", entry) for entry in same_scoring)], 1
        )
    ]
    if not references:
        raise ValueError(
            f"No calibrated grade {item.grade} items in '{QDRANT_COLLECTION}'; "
            "run 'Add the item parameters to the stored metadata'"
        )
    return {
        "exclude_keys": sorted(exclude_keys),
        "references": references,
        "system_prompt": parameter_system_prompt(item),
        "user_prompt": parameter_user_prompt(item, references, exclude_keys),
    }


def reference_summary(entry):
    """A reference item as stored with a prediction: its rank, key, similarity, percent correct, and parameters."""
    payload = entry["payload"]
    parameters = payload[ITEM_PARAMETERS_PAYLOAD]
    return {
        "rank": entry["rank"],
        "found_by": entry["found_by"],
        "item_key": entry["item_key"],
        "item_id": payload["item_id"],
        "assessment": payload["assessment"],
        "cosine_similarity": round(entry["cosine_similarity"], 4),
        "pct_correct_intl_avg": payload["pct_correct_intl_avg"],
        "irt_model": parameters["irt_model"],
        "scaled_points": parameters["scaled_points"],
        **{name: parameters[name] for name in model_parameters(parameters["irt_model"], parameters["scaled_points"])},
    }


def prediction_record(item, prepared):
    """Everything about a prediction except the LLM's output: the item, its reference items, and the prompts."""
    return {
        **item.model_dump(
            include={
                "item_id",
                "assessment",
                "grade",
                "subject",
                "content_domain",
                "main_topic",
                "cognitive_domain",
                "item_type",
                "max_points",
            }
        ),
        "scaled_points": item.points,
        "irt_model": item.irt_model,
        "target_calibration": CALIBRATIONS.get((item.assessment, item.grade)),
        "excluded_item_keys": prepared["exclude_keys"],
        "similar_items": [reference_summary(entry) for entry in prepared["references"]],
        "system_prompt": prepared["system_prompt"],
        "user_prompt": prepared["user_prompt"],
    }


def request_prediction(item, prepared, model=PARAMETER_LLM_MODEL, client=llm_client):
    """Send the prompts of `prepare_prediction` to the LLM. Returns its reasoning, parameters, and token usage.

    Output that does not fit the schema raises pydantic's ValidationError, output cut off at PARAMETER_MAX_TOKENS
    raises openai.LengthFinishReasonError, and no output at all raises NoPredictionError.
    """
    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": prepared["system_prompt"]},
            {"role": "user", "content": prepared["user_prompt"]},
        ],
        response_format=prediction_schema(item.irt_model, item.points),
        max_tokens=PARAMETER_MAX_TOKENS,
        extra_body={"reasoning": {"effort": PARAMETER_REASONING_EFFORT}, "usage": {"include": True}},
    )
    choice = completion.choices[0]
    if choice.message.parsed is None:
        raise NoPredictionError(f"No prediction (finish_reason={choice.finish_reason}): {choice.message.refusal}")
    output = choice.message.parsed.model_dump()
    names = predicted_parameters(item.irt_model, item.points)
    parameters = {name: output[name] for name in names}
    if item.irt_model == "GPCM":
        parameters[f"step{item.points}"] = -sum(parameters[f"step{step}"] for step in range(1, item.points))
    usage = completion.usage
    details = usage.completion_tokens_details
    return {
        "llm_model": model,
        "reasoning_effort": PARAMETER_REASONING_EFFORT,
        "reasoning": output["reasoning"],
        **{f"{name}_reasoning": output[f"{name}_reasoning"] for name in names},
        **parameters,
        # The reasoning summary that OpenRouter returns for a reasoning model, if any.
        "llm_thinking": getattr(choice.message, "reasoning", None),
        "finish_reason": choice.finish_reason,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "reasoning_tokens": details.reasoning_tokens if details else None,
        "cost_usd": getattr(usage, "cost", None),
        "predicted_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


def predict_item_parameters(item, exclude_keys=(), model=PARAMETER_LLM_MODEL, client=llm_client):
    """Predict the IRT parameters of a TimssItem with an LLM, using similar calibrated items as context.

    Returns the prediction record: the item, its reference items, the prompts, and the LLM's reasoning and parameters.
    """
    prepared = prepare_prediction(item, exclude_keys)
    return prediction_record(item, prepared) | request_prediction(item, prepared, model, client)

### Example: a new item

A made-up grade 4 multiple-choice item, predicted on the scale of the TIMSS 2011 calibration.
It costs one embedding request and one LLM request.

In [12]:
new_item = TimssItem(
    assessment="TIMSS 2011",
    grade=4,
    subject="Mathematics",
    content_domain="Number",
    main_topic="Whole Numbers",
    cognitive_domain="Applying",
    item_type="multiple_choice",
    stem="Maria had 52 stickers. She gave 18 stickers to her brother. How many stickers does Maria have now?",
    options=[
        {"label": "A", "text": "34"},
        {"label": "B", "text": "44"},
        {"label": "C", "text": "46"},
        {"label": "D", "text": "70"},
    ],
    correct_option="A",
)
print(f"Most similar calibrated grade {new_item.grade} items (the new item is scored with the {new_item.irt_model} model):")
display(similar_items_table(find_similar_items(new_item, limit=5)).dropna(axis=1, how="all").round(3))
print(f"System prompt:\n\n{parameter_system_prompt(new_item)}\n")

new_item_prediction = predict_item_parameters(new_item)
names = model_parameters(new_item.irt_model, new_item.points)
display(pd.DataFrame([{name: new_item_prediction[name] for name in names}], index=["predicted"]).round(3))
print(f"Reasoning: {new_item_prediction['reasoning']}")
for name in predicted_parameters(new_item.irt_model, new_item.points):
    print(f"{PARAMETER_LABELS[name]}: {new_item_prediction[f'{name}_reasoning']}")

Most similar calibrated grade 4 items (the new item is scored with the 3PL model):


,rank,cosine_similarity,assessment,item_id,content_domain,cognitive_domain,item_type,pct_correct_intl_avg,irt_model,scaled_points,slope,location,guessing
0,1,0.824,TIMSS 2011,M031187,Number,Applying,multiple_choice,73.0,3PL,1,0.679,-0.897,0.198
1,2,0.865,TIMSS 2011,M031317,Number,Knowing,multiple_choice,39.0,3PL,1,1.278,0.641,0.147
2,3,0.848,TIMSS 2007,M031173,Number,Applying,multiple_choice,57.6,3PL,1,1.559,-0.146,0.125
3,4,0.861,TIMSS 2003,M031220,Patterns and Relationships,Using Concepts,multiple_choice,73.0,3PL,1,0.797,-1.043,0.132
4,5,0.770,TIMSS 2007,M031134,Data Display,Applying,constructed_response,27.7,2PL,1,0.570,0.851,NaN


System prompt:

You are an expert in testing mathematics for fourth-grade students. You are very familiar with assessing these students in the Number content domain, especially Whole Numbers, and in the Applying cognitive domain, as the TIMSS 2011 framework defines them. You have written, reviewed, and analyzed many mathematics items for these students, and you know from experience how students across the TIMSS countries perform on them.

You are also a measurement expert in item response theory (IRT). You know the three-parameter logistic (3PL), two-parameter logistic (2PL), and generalized partial credit (GPCM) models, how TIMSS calibrates items with them, and how content, cognitive demand, the number and difficulty of steps, the numbers involved, reading load, figures, response format, distractor plausibility, and scoring rules shape item parameters.

Your task is to predict the IRT parameters that a new item would receive in a TIMSS calibration, using calibrated reference items of 

,slope,location,guessing
predicted,0.95,-0.75,0.17


Reasoning: The new item is a straightforward, single-step word problem requiring subtraction of two-digit whole numbers with regrouping (52 − 18 = 34). It is most comparable to other simple word problems and number sentence items in fourth grade, such as M041107 (12 − □ = 9), M031187 (Pete and pencils), and M031220 (Rob's apples). While those reference items test symbolic representation or simple concepts without arithmetic difficulty, the new item requires performing subtraction with borrowing. The distractors directly target typical arithmetic errors (D adds the numbers to get 70; B and C target common regrouping errors), making it slightly more challenging than M041107 (b = -1.328) but still easier than multi-step problems like M031173 (b = -0.146).
slope (a): The item is clear, concise, and features highly diagnostic distractors that clearly separate students who have mastered two-digit subtraction with regrouping from those with fundamental procedural misconceptions. This leads to

## Predict every calibrated item as if it were new

Every item in the collection that has parameters is predicted with `predict_item_parameters` as if it were new, and the prediction is compared with the item's calibrated parameters.

- The item and its other parts are left out of its context, both as reference items and from the calibration summary (`item_parts`). The other parts are the items of the same assessment and grade whose IDs differ only in a final part letter (`base_item_id`), such as M031346B and M031346C for M031346A, V4B for V4A, or T02B for T02A, and the other items of the same task (below).
- The item's own percent correct is not in its prompt.
- Each item is predicted with the IRT model and points of its calibration, so K14 and L18 of TIMSS Advanced 1995 are predicted as 1-point (2PL) items, and its `assessment` makes the prediction target the scale of its own calibration.
- Each item reuses its stored vector, so the run makes no embedding requests.

Some items belong to a task of several items built around one situation, but have IDs of their own, so their IDs do not show that they belong together.
For example, M031379 and M031380 continue the Trading Cards situation of M031346A to M031346C on a page headed "Trading Cards (Continued)", so when M031346A is predicted, M031346B, M031346C, M031379, and M031380 are all left out.
`SHARED_STIMULUS_TASKS` lists these tasks, found by comparing the stems and labels of the items of each assessment (shared passages, and labels such as "Trading Cards_Cartoon cards" and "Trading Cards_Trading sports cards"):

| Assessment | Grade | Task | Items |
| --- | --- | --- | --- |
| TIMSS 2003 | 4 | Bar graph of the bottles each class collected | M011009, M011012 |
| TIMSS 2003 | 4 | Number tiles | M031344A to M031344C, M031345A to M031345C |
| TIMSS 2003 | 8 | Geometry tiling | M032743, M032744, M032745 |
| TIMSS 2003 | 8 | Phone plans | M032762, M032763, M032764 |
| TIMSS 2007 | 8 | Class trip | M032753A to M032753C, M032754, M032755, M032756 |
| TIMSS 2011 | 4 | Trading cards | M031346A to M031346C, M031379, M031380 |
| TIMSS 2011 | 8 | Red and black tiles | M032757, M032760A to M032760C, M032761 |

Items that merely share wording, such as the coordinate-grid items M031088 and M041160A of TIMSS 2011 grade 4, are not tasks.

The first cell finds the reference items and writes the prompts of every item, which needs no API calls, and compares them with the saved predictions.
The second cell requests the predictions that are missing, or whose prompts, LLM, or reasoning effort changed since they were saved, `PREDICTION_WORKERS` at a time.
Each prediction is saved as it arrives in `data/cache/parameter_predictions/timss_math/<LLM>/`, one JSON file per item with the item, its reference items, the prompts, the LLM's reasoning and parameters, and the token usage and cost.
So an interrupted run resumes where it stopped, and a rerun with unchanged prompts sends no requests.
An item whose output does not fit the schema is reported and left for a rerun, and an API error (such as exhausted credit) stops the cell; the client retries a failed request twice before that.

In [13]:
PREDICTION_WORKERS = 8  # concurrent LLM requests
PREDICTIONS_CACHE_DIR = DATA_DIR / "cache" / "parameter_predictions" / "timss_math" / PARAMETER_LLM_MODEL.replace("/", "_")
PREDICTIONS_PATH = DATA_DIR / "timss_math_predicted_item_parameters.xlsx"


# Tasks of several items around one situation whose items have IDs of their own: assessment, grade, and the base IDs
# of the task's items (see the table above).
SHARED_STIMULUS_TASKS = [
    ("TIMSS 2003", 4, ["M011009", "M011012"]),  # bar graph of bottles
    ("TIMSS 2003", 4, ["M031344", "M031345"]),  # number tiles
    ("TIMSS 2003", 8, ["M032743", "M032744", "M032745"]),  # geometry tiling
    ("TIMSS 2003", 8, ["M032762", "M032763", "M032764"]),  # phone plans
    ("TIMSS 2007", 8, ["M032753", "M032754", "M032755", "M032756"]),  # class trip
    ("TIMSS 2011", 4, ["M031346", "M031379", "M031380"]),  # trading cards
    ("TIMSS 2011", 8, ["M032757", "M032760", "M032761"]),  # red and black tiles
]


def base_item_id(item_id):
    """The item ID without its part letter: M031346 for M031346A, V4 for V4A, T02 for T02B."""
    return re.sub(r"(?<=\d)[A-Z]$", "", item_id)


def prediction_cache_path(key):
    return PREDICTIONS_CACHE_DIR / f"{re.sub(r'[^A-Za-z0-9]+', '_', key)}.json"


def saved_prediction(key):
    """The saved prediction of the item with this key, if PARAMETER_LLM_MODEL made it from the current prompts with
    PARAMETER_REASONING_EFFORT."""
    path = prediction_cache_path(key)
    if not path.exists():
        return None
    record = json.loads(path.read_text(encoding="utf-8"))
    prepared = loo_prepared[key]
    is_current = (
        record["llm_model"] == PARAMETER_LLM_MODEL
        and record["reasoning_effort"] == PARAMETER_REASONING_EFFORT
        and record["system_prompt"] == prepared["system_prompt"]
        and record["user_prompt"] == prepared["user_prompt"]
    )
    return record if is_current else None


# All parts of each item, the item included: the collection items of its assessment and grade whose IDs differ only in
# the part letter, and the other items of its task.
task_ids = {
    (assessment, grade, base_id): base_ids[0]
    for assessment, grade, base_ids in SHARED_STIMULUS_TASKS
    for base_id in base_ids
}
chunk_bases = list(zip(chunks_table["assessment"], chunks_table["grade"], chunks_table["item_id"].map(base_item_id)))
unknown_task_items = sorted(set(task_ids) - set(chunk_bases))
if unknown_task_items:
    raise ValueError(f"SHARED_STIMULUS_TASKS lists items that are not in the collection: {unknown_task_items}")
parts_by_group = {}
for key, (assessment, grade, base_id) in zip(chunks_table["item_key"], chunk_bases):
    group = (assessment, grade, task_ids.get((assessment, grade, base_id), base_id))
    parts_by_group.setdefault(group, []).append(key)
item_parts = {key: sorted(parts) for parts in parts_by_group.values() for key in parts}

started = time.monotonic()
loo_items = {}
loo_prepared = {}
for key, row in calibrated_table.iterrows():
    item = TimssItem.model_validate(items_by_key[key] | {"scaled_points": int(row["scaled_points"])})
    if item.irt_model != row["irt_model"]:
        raise ValueError(f"{key} would be predicted with {item.irt_model} but is calibrated with {row['irt_model']}")
    loo_items[key] = item
    loo_prepared[key] = prepare_prediction(item, exclude_keys=item_parts[key])
print(f"Prepared {len(loo_items)} items in {time.monotonic() - started:.0f} s")

multi_part = [key for key in loo_items if len(item_parts[key]) > 1]
print(f"{len(multi_part)} items have other parts, which their predictions leave out; for example:")
for key in ["TIMSS 2011/grade 4/Mathematics/M031346A", "TIMSS 2003/grade 8/Mathematics/M032743"]:
    print(f"  {key}: {', '.join(items_by_key[part]['item_id'] for part in item_parts[key] if part != key)}")

context = pd.DataFrame(
    {
        "grade": [item.grade for item in loo_items.values()],
        "reference_items": [len(prepared["references"]) for prepared in loo_prepared.values()],
        "prompt_characters": [
            len(prepared["system_prompt"]) + len(prepared["user_prompt"]) for prepared in loo_prepared.values()
        ],
    }
)
display(context.groupby("grade").agg(["mean", "min", "max"]).round())

status = pd.Series(
    [
        "saved"
        if saved_prediction(key)
        else "outdated (prompts, LLM, or reasoning effort changed)"
        if prediction_cache_path(key).exists()
        else "not predicted yet"
        for key in loo_items
    ]
)
print(f"Predictions with {PARAMETER_LLM_MODEL}:")
display(status.value_counts().rename("items").to_frame())

Prepared 722 items in 18 s
111 items have other parts, which their predictions leave out; for example:
  TIMSS 2011/grade 4/Mathematics/M031346A: M031346B, M031346C, M031379, M031380
  TIMSS 2003/grade 8/Mathematics/M032743: M032744, M032745


reference_items         prompt_characters              
                 mean min max              mean    min    max
grade                                                        
4                10.0  10  15           18636.0  11131  35760
8                11.0  10  15           17761.0  10835  34006
12               11.0  10  14           21069.0  11307  34089

Predictions with google/gemini-3.8-flash:


,items
not predicted yet,718
saved,4


In [14]:
PREDICTIONS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
pending = [key for key in loo_items if saved_prediction(key) is None]
print(f"Requesting {len(pending)} of {len(loo_items)} predictions from {PARAMETER_LLM_MODEL}, {PREDICTION_WORKERS} at a time")
loo_unusable = {}  # item key -> error, for output that could not be used
run_cost = 0.0
started = time.monotonic()
executor = concurrent.futures.ThreadPoolExecutor(PREDICTION_WORKERS)
futures = {executor.submit(request_prediction, loo_items[key], loo_prepared[key]): key for key in pending}
try:
    for done, future in enumerate(concurrent.futures.as_completed(futures), 1):
        key = futures[future]
        try:
            output = future.result()
        except (ValidationError, openai.LengthFinishReasonError, openai.ContentFilterFinishReasonError, NoPredictionError) as error:
            loo_unusable[key] = f"{type(error).__name__}: {error}"
            print(f"{key}: unusable output ({type(error).__name__}), left for a rerun")
            continue
        record = prediction_record(loo_items[key], loo_prepared[key]) | output
        prediction_cache_path(key).write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
        run_cost += output["cost_usd"] or 0.0
        if done % 50 == 0 or done == len(pending):
            print(f"{done}/{len(pending)} in {time.monotonic() - started:.0f} s, ${run_cost:.2f}")
finally:
    # On an API error or an interrupt, send no further requests. The predictions so far are saved.
    executor.shutdown(wait=False, cancel_futures=True)

saved = sum(saved_prediction(key) is not None for key in loo_items)
print(f"{saved} of {len(loo_items)} items are predicted; this run cost ${run_cost:.2f}")
if loo_unusable:
    print(f"Rerun this cell to request the {len(loo_unusable)} items with unusable output again")

Requesting 718 of 722 predictions from google/gemini-3.8-flash, 8 at a time
50/718 in 97 s, $0.58
100/718 in 185 s, $1.20
150/718 in 270 s, $1.79
200/718 in 360 s, $2.40
250/718 in 451 s, $2.99
300/718 in 541 s, $3.58
350/718 in 624 s, $4.18
400/718 in 703 s, $4.75
450/718 in 784 s, $5.31
500/718 in 865 s, $5.91
550/718 in 943 s, $6.51
600/718 in 1022 s, $7.06
650/718 in 1106 s, $7.67
700/718 in 1197 s, $8.25
718/718 in 1228 s, $8.48
722 of 722 items are predicted; this run cost $8.48


### Prediction errors

`predictions` has one row per item, with every column of the prediction process:

- The item: key, assessment, grade, subject, ID, label, content domain, main topic, cognitive domain, format, maximum points and points in the scaling, IRT model, and the calibration of its parameters.
- Its context: the other parts left out (`excluded_item_ids`, including the other items of its task), the number of reference items and their IDs in prompt order (`similar_item_ids`), and `similar_items`, which gives, as JSON, each reference item's rank, how it was found (among the `N_SIMILAR` most similar items, or among those with the same model), assessment, cosine similarity, percent correct, and calibrated parameters.
- The LLM's output: `reasoning`, its comparison of the item with the reference items, and its reasoning for each parameter it predicted (`slope_reasoning` and so on), plus `llm_thinking`, the reasoning summary the model returned, if any.
- For each parameter: the predicted value (`slope_predicted`), the calibrated value (`slope_calibrated`), the prediction error, predicted minus calibrated (`slope_error`), and its absolute value (`slope_abs_error`). Parameters that the item's model lacks are blank. The last step of a GPCM item is derived from the others, so its error mirrors theirs.
- The LLM and its reasoning effort, finish reason, token counts, cost in US dollars, time of the prediction (UTC), both prompts, and the name of the file in `data/cache/parameter_predictions/timss_math/<LLM>/` that the prediction is saved in.

`predictions` is exported to `data/timss_math_predicted_item_parameters.xlsx` (sheet `predictions`), with the error summaries below in sheets of their own.
An Excel cell holds at most 32,767 characters, so a longer text, such as a long prompt, is cut there in the workbook; the saved prediction has it in full.

In [15]:
records = {key: saved_prediction(key) for key in loo_items}
not_predicted = [key for key, record in records.items() if record is None]
if not_predicted:
    print(f"{len(not_predicted)} items have no current prediction and are left out; rerun the cell above for them")
records = {key: record for key, record in records.items() if record is not None}


def prediction_row(key, record):
    """One row of `predictions`: the item, its context, each parameter with its error, and the LLM's output."""
    calibrated = calibrated_table.loc[key]
    row = {
        "item_key": key,
        **{column: record[column] for column in ("assessment", "grade", "subject", "item_id")},
        "item_label": items_by_key[key]["item_label"],
        **{
            column: record[column]
            for column in (
                "content_domain",
                "main_topic",
                "cognitive_domain",
                "item_type",
                "max_points",
                "scaled_points",
                "irt_model",
            )
        },
        "calibration": calibrated["calibration"],
        "excluded_item_ids": ", ".join(items_by_key[part]["item_id"] for part in record["excluded_item_keys"] if part != key),
        "n_similar_items": len(record["similar_items"]),
        "similar_item_ids": ", ".join(f"{entry['item_id']} ({entry['assessment']})" for entry in record["similar_items"]),
    }
    for name in PARAMETER_NAMES:
        predicted = np.nan if record.get(name) is None else float(record[name])
        error = predicted - calibrated[name]  # NaN for parameters the item's model lacks
        row |= {
            f"{name}_predicted": predicted,
            f"{name}_calibrated": calibrated[name],
            f"{name}_error": error,
            f"{name}_abs_error": abs(error),
        }
    return row | {
        "llm_model": record["llm_model"],
        "reasoning_effort": record["reasoning_effort"],
        "reasoning": record["reasoning"],
        **{f"{name}_reasoning": record.get(f"{name}_reasoning") for name in PARAMETER_NAMES},
        **{
            column: record[column]
            for column in (
                "llm_thinking",
                "finish_reason",
                "prompt_tokens",
                "completion_tokens",
                "reasoning_tokens",
                "cost_usd",
                "predicted_at",
            )
        },
        "similar_items": json.dumps(record["similar_items"], ensure_ascii=False),
        "system_prompt": record["system_prompt"],
        "user_prompt": record["user_prompt"],
        "prediction_file": prediction_cache_path(key).name,
    }


predictions = pd.DataFrame([prediction_row(key, record) for key, record in records.items()])
predicted_kinds = predictions[[f"{name}_predicted" for name in PARAMETER_NAMES]].notna().to_numpy()
calibrated_kinds = predictions[[f"{name}_calibrated" for name in PARAMETER_NAMES]].notna().to_numpy()
mismatched = predictions.loc[(predicted_kinds != calibrated_kinds).any(axis=1), "item_key"].tolist()
if mismatched:
    raise ValueError(f"Predicted and calibrated items have different parameters: {mismatched}")

print(f"{len(predictions)} items predicted with {predictions['llm_model'].iloc[0]}, at ${predictions['cost_usd'].sum():.2f}")
display(
    predictions[["assessment", "item_id", "irt_model", "excluded_item_ids", "location_predicted", "location_calibrated", "location_error"]]
    .head()
    .round(3)
)

722 items predicted with google/gemini-3.8-flash, at $8.54


,assessment,item_id,irt_model,excluded_item_ids,location_predicted,location_calibrated,location_error
0,TIMSS 1995,I3,3PL,,0.36,0.111,0.249
1,TIMSS 1995,I4,3PL,,-1.15,-1.368,0.218
2,TIMSS 1995,I9,3PL,,0.12,-0.297,0.417
3,TIMSS 1995,J4,3PL,,0.25,0.521,-0.271
4,TIMSS 1995,J9,3PL,,-0.12,-0.761,0.641


### Prediction error by grade

The summary covers the parameters that the LLM predicts, for every item whose model has them; the last step of a GPCM item is derived, so it is left out.
For each grade and parameter, it gives:

- `items`: the number of items.
- `mean_error`: the bias, the mean of predicted minus calibrated values.
- `sd_error`, `mean_absolute_error`, and `rmse` (root mean squared error).
- `correlation`: the correlation of the predicted with the calibrated values.
- Two baselines that need no LLM, as mean absolute errors. `baseline_mae_calibration_mean` predicts each item's parameter by the mean in its calibration summary (the calibrated items of its grade with its model and points, its parts left out), and `baseline_mae_reference_mean` by the mean over its reference items with its model and points. A useful LLM beats both.

The `all` rows pool the grades.
The same summary by assessment shows whether the items of one calibration are predicted with a bias, as a difference between the scales of the calibrations would cause.
`prediction_errors` has one row per item and predicted parameter, with both baselines; it is exported too.

In [16]:
error_rows = []
for key, record in records.items():
    item = loo_items[key]
    prior = same_scoring_items(item, exclude_keys=record["excluded_item_keys"])
    same_scoring_references = [
        entry
        for entry in record["similar_items"]
        if entry["irt_model"] == item.irt_model and entry["scaled_points"] == item.points
    ]
    for name in predicted_parameters(item.irt_model, item.points):
        error_rows.append(
            {
                "item_key": key,
                "assessment": item.assessment,
                "grade": item.grade,
                "item_id": item.item_id,
                "irt_model": item.irt_model,
                "parameter": name,
                "predicted": record[name],
                "calibrated": calibrated_table.at[key, name],
                "calibration_mean": prior[name].mean(),
                "reference_mean": np.mean([entry[name] for entry in same_scoring_references])
                if same_scoring_references
                else np.nan,
            }
        )
prediction_errors = pd.DataFrame(error_rows)
prediction_errors["error"] = prediction_errors["predicted"] - prediction_errors["calibrated"]
prediction_errors["parameter"] = pd.Categorical(prediction_errors["parameter"], categories=PARAMETER_NAMES, ordered=True)


def summarize_errors(rows):
    error = rows["predicted"] - rows["calibrated"]
    baseline_mae = lambda baseline: (rows[baseline] - rows["calibrated"]).abs().mean()
    return pd.Series(
        {
            "items": len(rows),
            "mean_error": error.mean(),
            "sd_error": error.std(),
            "mean_absolute_error": error.abs().mean(),
            "rmse": np.sqrt((error**2).mean()),
            "correlation": rows["predicted"].corr(rows["calibrated"]) if len(rows) > 2 else np.nan,
            "baseline_mae_calibration_mean": baseline_mae("calibration_mean"),
            "baseline_mae_reference_mean": baseline_mae("reference_mean"),
        }
    )


def error_summary(errors, by):
    """summarize_errors for each value of `by` and each parameter, then for all values together."""
    columns = ["predicted", "calibrated", "calibration_mean", "reference_mean"]
    summary = pd.concat(
        [
            errors.groupby([by, "parameter"], observed=True)[columns].apply(summarize_errors),
            errors.assign(**{by: "all"}).groupby([by, "parameter"], observed=True)[columns].apply(summarize_errors),
        ]
    )
    return summary.astype({"items": int})


summary_by_grade = error_summary(prediction_errors, "grade")
summary_by_assessment = error_summary(prediction_errors, "assessment")
print("Prediction errors (predicted minus calibrated) by grade:")
display(summary_by_grade.round(3))
print("By assessment:")
display(summary_by_assessment.round(3))

Prediction errors (predicted minus calibrated) by grade:


items  mean_error  sd_error  mean_absolute_error   rmse  \
grade parameter                                                            
4     slope        295       0.049     0.245                0.202  0.250   
      location     295       0.010     0.548                0.417  0.547   
      guessing     164      -0.016     0.066                0.052  0.068   
      step1         18       0.353     1.032                0.736  1.063   
8     slope        356       0.044     0.287                0.225  0.290   
      location     356      -0.101     0.459                0.366  0.469   
      guessing     234      -0.016     0.073                0.056  0.074   
      step1         30       0.104     0.970                0.828  0.960   
12    slope         71       0.036     0.292                0.210  0.292   
      location      71      -0.152     0.509                0.431  0.528   
      guessing      44      -0.007     0.086                0.067  0.085   
      step1          8      -0.028     0.693                0.524  0.649   
      step2          3      -0.082     0.799                0.581  0.657   
all   slope        722       0.046     0.271                0.214  0.275   
      location     722      -0.060     0.505                0.393  0.508   
      guessing     442      -0.015     0.072                0.056  0.073   
      step1         56       0.165     0.952                0.755  0.958   
      step2          3      -0.082     0.799                0.581  0.657   

                 correlation  baseline_mae_calibration_mean  \
grade parameter                                               
4     slope            0.502                          0.220   
      location         0.704                          0.617   
      guessing         0.215                          0.053   
      step1            0.434                          0.961   
8     slope            0.571                          0.269   
      location         0.685                          0.486   
      guessing         0.236                          0.060   
      step1           -0.116                          0.743   
12    slope            0.573                          0.235   
      location         0.547                          0.500   
      guessing         0.204                          0.071   
      step1           -0.542                          0.468   
      step2           -0.889                          0.720   
all   slope            0.586                          0.245   
      location         0.719                          0.541   
      guessing         0.230                          0.058   
      step1            0.384                          0.774   
      step2           -0.889                          0.720   

                 baseline_mae_reference_mean  
grade parameter                               
4     slope                            0.215  
      location                         0.584  
      guessing                         0.053  
      step1                            1.096  
8     slope                            0.243  
      location                         0.483  
      guessing                         0.062  
      step1                            0.832  
12    slope                            0.227  
      location                         0.510  
      guessing                         0.071  
      step1                            0.468  
      step2                            0.720  
all   slope                            0.230  
      location                         0.527  
      guessing                         0.060  
      step1                            0.865  
      step2                            0.720

By assessment:


items  mean_error  sd_error  \
assessment          parameter                                
TIMSS 1995          slope         69       0.059     0.258   
                    location      69       0.116     0.579   
                    guessing      41      -0.017     0.066   
                    step1          6       0.074     1.005   
TIMSS 1999          slope         81       0.095     0.266   
                    location      81      -0.037     0.450   
                    guessing      64      -0.014     0.077   
                    step1          3       0.177     0.886   
TIMSS 2003          slope        178       0.117     0.223   
                    location     178      -0.028     0.551   
                    guessing     123      -0.020     0.074   
                    step1          9       0.621     1.338   
TIMSS 2007          slope        162       0.028     0.246   
                    location     162      -0.081     0.465   
                    guessing      86      -0.000     0.056   
                    step1         14      -0.063     0.871   
TIMSS 2011          slope        161      -0.043     0.314   
                    location     161      -0.123     0.465   
                    guessing      84      -0.026     0.072   
                    step1         16       0.237     0.917   
TIMSS Advanced 1995 slope         31       0.029     0.297   
                    location      31      -0.234     0.517   
                    guessing      19      -0.027     0.096   
                    step1          6      -0.150     0.771   
                    step2          3      -0.082     0.799   
TIMSS Advanced 2008 slope         40       0.041     0.291   
                    location      40      -0.088     0.500   
                    guessing      25       0.009     0.075   
                    step1          2       0.337     0.184   
all                 slope        722       0.046     0.271   
                    location     722      -0.060     0.505   
                    guessing     442      -0.015     0.072   
                    step1         56       0.165     0.952   
                    step2          3      -0.082     0.799   

                               mean_absolute_error   rmse  correlation  \
assessment          parameter                                            
TIMSS 1995          slope                    0.209  0.262        0.390   
                    location                 0.457  0.586        0.661   
                    guessing                 0.047  0.067        0.349   
                    step1                    0.702  0.921        0.572   
TIMSS 1999          slope                    0.216  0.281        0.572   
                    location                 0.358  0.449        0.659   
                    guessing                 0.060  0.078        0.230   
                    step1                    0.690  0.745        0.466   
TIMSS 2003          slope                    0.204  0.252        0.617   
                    location                 0.418  0.550        0.737   
                    guessing                 0.060  0.076        0.282   
                    step1                    1.108  1.406       -0.081   
TIMSS 2007          slope                    0.188  0.246        0.556   
                    location                 0.365  0.470        0.662   
                    guessing                 0.044  0.056        0.194   
                    step1                    0.724  0.841       -0.078   
TIMSS 2011          slope                    0.254  0.316        0.634   
                    location                 0.369  0.479        0.780   
                    guessing                 0.057  0.076        0.055   
                    step1                    0.731  0.919        0.526   
TIMSS Advanced 1995 slope                    0.199  0.294        0.570   
                    location                 0.459  0.560        0.533   
                    gu

In [17]:
EXCEL_CELL_LIMIT = 32_767


def excel_value(value):
    # Excel rejects control characters and holds at most EXCEL_CELL_LIMIT characters in a cell.
    if not isinstance(value, str):
        return value
    value = ILLEGAL_CHARACTERS_RE.sub("", value)
    if len(value) <= EXCEL_CELL_LIMIT:
        return value
    note = " [cut at the Excel cell limit; the saved prediction has the full text]"
    return value[: EXCEL_CELL_LIMIT - len(note)] + note


with pd.ExcelWriter(PREDICTIONS_PATH) as writer:
    predictions.map(excel_value).to_excel(writer, sheet_name="predictions", index=False)
    summary_by_grade.reset_index().to_excel(writer, sheet_name="summary_by_grade", index=False)
    summary_by_assessment.reset_index().to_excel(writer, sheet_name="summary_by_assessment", index=False)
    prediction_errors.to_excel(writer, sheet_name="prediction_errors", index=False)
cut = predictions.map(lambda value: isinstance(value, str) and len(value) > EXCEL_CELL_LIMIT).to_numpy().sum()
print(f"Exported {len(predictions)} predictions to {PREDICTIONS_PATH} ({cut} cells cut at the Excel cell limit)")

Exported 722 predictions to /Users/ze/Documents/projects/timss-1/notebooks/data/timss_math_predicted_item_parameters.xlsx (0 cells cut at the Excel cell limit)


### Results of the run on 2026-10-04

`google/gemini-3.8-flash` at medium reasoning effort predicted all 722 items in about 20 minutes, with 8 concurrent requests, for $8.54; every output fit the schema.

- Location: the mean absolute error is 0.39 over all items (grade 4: 0.42, grade 8: 0.37, grade 12: 0.43), against 0.54 for the calibration-summary mean and 0.53 for the reference-item mean, and the predictions correlate 0.72 with the calibrated locations.
- Slope: the mean absolute error is 0.21, against 0.25 and 0.23 for the two baselines (correlation 0.59).
- Guessing: the mean absolute error is 0.056, no better than the baselines (0.058 and 0.060).
- GPCM steps: the step 1 errors (mean absolute error 0.76 over 56 items) are no better than the baselines, and the correlation is negative at grades 8 and 12, so the predicted steps carry little information.
- By assessment, the location bias is small for most calibrations, but the TIMSS 2011 items are predicted 0.12 and the TIMSS Advanced 1995 items 0.23 easier than calibrated, on average.